# 03 - Baseline Fault Classification

Train a Random Forest baseline on the canonical MachineFeatureVector emitted by `02_feature_engineering.ipynb`, and evaluate on a held-out **recording-level** test split.

Pipeline:

```
cwru_features.parquet
    -> group-aware train / validation / test split (by recording_id)
    -> RandomForestClassifier (fit on train, tuned against validation)
    -> classification report + confusion matrix on test
```

This notebook contains **no** hard-coded accuracy numbers. Results depend on which recordings have been downloaded and are computed live at run time.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'ml').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'ml' / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'ml' / 'models'
FEATURES_PATH = PROCESSED_DIR / 'cwru_features.parquet'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd

if not FEATURES_PATH.is_file():
    raise SystemExit(
        f'Feature parquet not found at {FEATURES_PATH}. '
        'Run 02_feature_engineering.ipynb first.'
    )

features_df = pd.read_parquet(FEATURES_PATH)
features_df.head()

In [ ]:
from ml.src.split_dataset import group_train_val_test_split, summarize_split

split = group_train_val_test_split(features_df, validation_size=0.15, test_size=0.15, random_state=42)
summarize_split(split)

In [ ]:
from ml.src.train_baseline import train_and_persist

result = train_and_persist(split.train, split.validation, MODELS_DIR)
print(f'saved model -> {result.model_path}')
print(f'validation accuracy: {result.validation_accuracy:.4f}')
print(f'validation macro-F1: {result.validation_macro_f1:.4f}')

In [ ]:
from ml.src.evaluate import evaluate

report = evaluate(result.model_path, split.test)
print(f'test accuracy: {report.accuracy:.4f}')
print(f'test macro-F1: {report.macro_f1:.4f}')
print()
print(report.report_text)

In [ ]:
report.confusion